# Data Transform

In this notebook, we will ask you a series of questions to evaluate your findings from your EDA. Based on your response & justification, we will ask you to also apply a subsequent data transformation. 

If you state that you will not apply any data transformations for this step, you must **justify** as to why your dataset/machine-learning does not require the mentioned data preprocessing step.

The bonus step is completely optional, but if you provide a sufficient feature engineering step in this project we will add `1000` points to your Kahoot leaderboard score.

You will write out this transformed dataframe as a `.csv` file to your `data/` folder.

**Note**: Again, note that this dataset is quite large. If you find that some data operations take too long to complete on your machine, simply use the `sample()` method to transform a subset of your data.

In [4]:
import pandas as pd
import numpy as np
import datetime
from sklearn.preprocessing import OneHotEncoder


## Q1

Does your model contain any missing values or "non-predictive" columns? If so, which adjustments should you take to ensure that your model has good predictive capabilities? Apply your data transformations (if any) in the code-block below.

My model did not contain any missing values but it did contain some non-predictive columns such as nameDest and nameOrig. Best course of action is to not include these columns in the model.

In [ ]:
# Load data
transactions = pd.read_csv("../data/bank_transactions.csv")

# Drop non-predictive columns
transactions_dropped = transactions.drop(['nameOrig', 'nameDest'], axis=1)

# Take a sample of 30,000 rows
transactions_dropped_sample = transactions_dropped.sample(n=30000, random_state=42)

# Verify
print("Sample shape:", transactions_dropped_sample.shape)
print("Missing values:\n", transactions_dropped_sample.isnull().sum())


Sample shape: (30000, 8)
Missing values:
 type              0
amount            0
oldbalanceOrg     0
newbalanceOrig    0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


## Q2

Do certain transaction types consistently differ in amount or fraud likelihood? If so, how might you transform the type column to make this pattern usable by a machine learning model? Apply your data transformations (if any) in the code-block below.

The transaction types differ in amount and fraud. Fraud is found specfically in TRANSFER and CASH_OUT types.

In [35]:
transactions = pd.read_csv("../data/bank_transactions.csv")

transactions.head()

,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,PAYMENT,983.09,C1454812978,36730.24,35747.15,M1491308340,0.00,0.00,0,0
1,PAYMENT,55215.25,C1031766358,99414.00,44198.75,M2102868029,0.00,0.00,0,0
2,CASH_IN,220986.01,C1451868666,7773074.97,7994060.98,C1339195526,924031.48,703045.48,0,0
3,TRANSFER,2357394.75,C458368123,0.00,0.00,C620979654,4202580.45,6559975.19,0,0
4,CASH_OUT,67990.14,C1098978063,0.00,0.00,C142246322,625317.04,693307.19,0,0


In [20]:
# Load data
transactions1 = pd.read_csv("../data/bank_transactions.csv")

sample_transactions = transactions1.sample(n=1000, random_state=42)


type_features = ["type"]                              
num_features = ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]    


X_cat = sample_transactions[type_features]
X_num = sample_transactions[num_features]


X_cat.head()

,type
987231,CASH_IN
79954,CASH_IN
567130,CASH_OUT
500891,CASH_OUT
55399,PAYMENT


In [21]:
X_num.head()


,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest
987231,54152.03,42823.36,96975.39,11438021.32,10695480.59
79954,279331.66,8385167.08,8664498.74,394276.66,114945.00
567130,185673.97,0.00,0.00,396994.01,582667.97
500891,128216.41,12158.00,0.00,17406313.64,17534530.05
55399,17567.71,104890.00,87322.29,0.00,0.00


In [22]:
ohe = OneHotEncoder()
X_cat_full = ohe.fit_transform(X_cat).toarray()


X_cat_full


array([[1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       ...,
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.]], shape=(1000, 5))

In [23]:
ohe.get_feature_names_out(['type'])


array(['type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT',
       'type_TRANSFER'], dtype=object)

In [24]:
cat_names = ohe.get_feature_names_out(['type'])

encoded_df = pd.DataFrame(X_cat_full, columns=cat_names, index=sample_transactions.index)

In [25]:
encoded_df.head()


,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
987231,1.0,0.0,0.0,0.0,0.0
79954,1.0,0.0,0.0,0.0,0.0
567130,0.0,1.0,0.0,0.0,0.0
500891,0.0,1.0,0.0,0.0,0.0
55399,0.0,0.0,0.0,1.0,0.0


In [26]:
full_df = pd.concat([X_num, encoded_df], axis=1)


full_df


,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
987231,54152.03,42823.36,96975.39,11438021.32,10695480.59,1.0,0.0,0.0,0.0,0.0
79954,279331.66,8385167.08,8664498.74,394276.66,114945.00,1.0,0.0,0.0,0.0,0.0
567130,185673.97,0.00,0.00,396994.01,582667.97,0.0,1.0,0.0,0.0,0.0
500891,128216.41,12158.00,0.00,17406313.64,17534530.05,0.0,1.0,0.0,0.0,0.0
55399,17567.71,104890.00,87322.29,0.00,0.00,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...
914398,387953.23,11197.00,399150.23,1671214.48,1283261.25,1.0,0.0,0.0,0.0,0.0
189053,156675.31,42634.00,0.00,2878.61,159553.92,0.0,0.0,0.0,0.0,1.0
47977,239839.86,29824.00,269663.86,0.00,0.00,1.0,0.0,0.0,0.0,0.0
600785,338257.37,61178.00,399435.37,598865.77,260608.40,1.0,0.0,0.0,0.0,0.0


In [27]:
ohe = OneHotEncoder(drop='first')
X_cat_full = ohe.fit_transform(X_cat).toarray()


X_cat_full


array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [1., 0., 0., 0.],
       ...,
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [1., 0., 0., 0.]], shape=(1000, 4))

## Bonus (optional)

Are there interaction effects between variables (e.g., fraud and high amount and transaction type) that aren't captured directly in the dataset? Would it be helpful to manually engineer any new features that reflect these interactions? Apply your data transformations (if any) in the code-block below.

Answer Here

In [2]:
# write out newly transformed dataset to your folder
...